# 02 · Preprocessing
Raw downloads → converted files ([01_data_download.ipynb](01_data_download.ipynb)) → QC / normalize / HVG / batch-correct / cell-type via `run_pipeline()` → training-ready arrays + MIL bags.

In [ ]:
import sys
sys.path.insert(0, '../src')
from preprocess import run_pipeline

CFG = '../configs/default.yaml'

## Run the pipeline
Sources listed in `configs/default.yaml` that haven't been downloaded/converted yet are skipped with a message rather than raising — run with whatever subset you already have.

In [ ]:
try:
    cell_data, bags = run_pipeline(CFG)
except ValueError as e:
    print(e)
    cell_data, bags = None, []

## Inspect outputs

In [ ]:
import numpy as np

if cell_data is not None:
    print(f"gene_matrix       : {cell_data['gene_matrix'].shape}")
    print(f"smoke class counts: "
          f"{dict(zip(*np.unique(cell_data['smoke_labels'], return_counts=True)))}")
    print(f"malignant fraction: {cell_data['malignancy_labels'].mean():.2%}")
    print(f"subject bags       : {len(bags)}")
    if bags:
        print(f"  e.g. {bags[0]['subject_id']}: {bags[0]['gene_matrix'].shape}")

Outputs are written to `data/processed/` (`gene_matrix.npy`, `smoke_labels.npy`, `malignancy_labels.npy`, `cell_type_ids.npy`, `cell_metadata.csv`) for `CellLevelDataset.from_dir()` in [03_training.ipynb](03_training.ipynb), plus the in-memory `bags` list for `SubjectLevelDataset`.